# Check 3+ with the paper's evaluation

The edge groups of check 3+ (`walk_coverage_analysis.ipynb`: training edges shown in the walks, missed by the walks with
the two nodes in the same walk, missed with the two nodes never in the same walk, and held-out test edges) evaluated
exactly as in the paper:

- **negatives**: one per positive, sampled with `negative_samples` of `netmedgpt.sentence_generation` (tails of the
  same relation in the KG, not linked to the head; same number per head as positives);
- **AUROC / AUPRC** over the positives and negatives of each group (as `link_pred_val`);
- **Recall@100** per head over all candidate tails of the relation (as `node_level_eval`).

For the held-out test edges this reproduces the benchmark evaluation, so the groups can be compared directly with the
paper's test values. Models: the saved benchmark checkpoints (NetMedGPT only). 5 seeds, random and zero-shot split.

In [ ]:
import os
import sys
if os.path.basename(os.getcwd()) == "revision_scripts":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import json
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, average_precision_score

from netmedgpt.model import TransformerModel
from netmedgpt.sentence_generation import negative_samples
from netmedgpt.utilities import sim, node_level_eval

RUNS = {"random_link_split": "netmedgpt_randomLinkSplit_seed{seed}", "zero_shot_split": "netmedgpt_zeroShot_seed{seed}"}
NODE_TYPES = {"indication": "disease|drug", "contraindication": "disease|drug", "off-label use": "disease|drug",
              "drug_protein": "drug|gene/protein", "drug_effect": "drug|effect/phenotype"}
GROUPS = {"seen": "training edge, shown in walks",
          "not adjacent, same walk": "training edge, missed by walks (nodes co-occur)",
          "not adjacent, never in same walk": "training edge, missed by walks (nodes never co-occur)",
          "test": "held-out test edge"}
SEEDS = [1, 2, 3, 4, 5]
CHECK3_DIR = "data/new_analysis/coverage"
CKPT_DIR = "data/model_checkpoints/saved_models_evaluation"
GPU = 1
OUT_DIR = "data/new_analysis/coverage"

In [ ]:
# ── KG and model (as other/evaluate_checkpoints.py) ───────────────────────────
all_param = json.load(open("data/parameters.json"))
hp = all_param["best_hyperparam"]
seq_len = all_param["node2vec"]["walk_length"] * 2 - 1
device = torch.device(f"cuda:{GPU}" if torch.cuda.is_available() else "cpu")
nodes = pd.read_csv("data/KG/nodes.csv")
edge = pd.read_csv("data/KG/edges.csv")
feat = torch.load("data/KG/embeddings.pt")
mask_token = int(edge["z_index"].max()) + 1
entity = list(nodes["node_type"].unique()) + list(edge["relation"].unique()) + ["mask"]
relation_mask_index = pd.concat([edge[["relation", "z_index"]].drop_duplicates(),
                                 pd.DataFrame([["mask", mask_token]], columns=["relation", "z_index"])], ignore_index=True)
rel_token = edge.drop_duplicates("relation").set_index("relation")["z_index"].astype(int).to_dict()

# negative pool and Recall@100 candidates per relation, as in sentence_generation / evaluate_checkpoints
tail_pool = {r: edge.loc[(edge["relation"] == r) & (edge["node_types"] == nt), "y_index"].unique() for r, nt in NODE_TYPES.items()}
type_ids = {t: torch.tensor(nodes.loc[nodes["node_type"] == t, "node_index"].unique(), dtype=torch.int64)
            for t in ["drug", "gene/protein", "effect/phenotype"]}
candidates = {r: type_ids[nt.split("|")[1]] for r, nt in NODE_TYPES.items()}


def load_model(path):
    ckpt = torch.load(path, map_location=device)
    state = ckpt["model_state"] if "model_state" in ckpt else ckpt     # two saved formats
    h, sl = (ckpt.get("hyperparams", hp), ckpt.get("seq_len", seq_len)) if "model_state" in ckpt else (hp, seq_len)
    model = TransformerModel(mask_token + 1, h["hidden_channels"], h["nhead"], h["N_encoder_layers"], sl, device=device,
                             feat=feat, nodes=nodes, entity=entity, relation_mask_index=relation_mask_index,
                             pos_emb="fixed").to(device)
    model.load_state_dict(state)
    return model.eval(), h, sl

In [ ]:
# ── Evaluate every group like the benchmark ──────────────────────────────────
rows = []
for inference, ckpt_name in RUNS.items():
    for seed in SEEDS:
        edges = pd.read_csv(os.path.join(CHECK3_DIR, f"{inference}_seed{seed}", "check3x_edges.csv"))
        edges = edges[(edges["method"] == "NetMedGPT") & edges["group"].isin(GROUPS)]
        model, h, sl = load_model(os.path.join(CKPT_DIR, ckpt_name.format(seed=seed) + ".pt"))
        for (rel, grp), g in edges.groupby(["relation", "group"]):
            pos = g.rename(columns={"head": "x_index", "tail": "y_index"})[["x_index", "y_index"]].drop_duplicates()
            neg = negative_samples(pos, edge, tail_pool[rel], seed)
            pairs = np.vstack([pos.to_numpy(), np.array(neg, dtype=np.int64).reshape(-1, 2)])
            label = np.r_[np.ones(len(pos)), np.zeros(len(neg))]
            z = rel_token[rel]
            idx = torch.tensor(np.vstack([pairs[:, 0], np.full(len(pairs), z), pairs[:, 1]]), dtype=torch.long)
            pred = sim(idx, model, batch_size=h["batch_size"], method="prob", device=device, mask_token=mask_token, seq_len=sl)
            recall, precision = node_level_eval(idx, torch.tensor(label), {z: candidates[rel]}, model,
                                                device=device, mask_token=mask_token, seq_len=sl)
            rows.append({"inference": inference, "seed": seed, "relation": rel, "group": GROUPS[grp],
                         "positives": len(pos), "AUROC": roc_auc_score(label, pred),
                         "AUPRC": average_precision_score(label, pred), "Recall@100": recall[0]})
        del model
        torch.cuda.empty_cache()
        print(f"{inference} seed {seed} done")

res = pd.DataFrame(rows)
res.to_csv(os.path.join(OUT_DIR, "check3x_paper_metrics_per_seed.csv"), index=False)

In [ ]:
# ── Mean over seeds ───────────────────────────────────────────────────────────
order = list(GROUPS.values())
summary = (res.groupby(["inference", "relation", "group"])[["positives", "AUROC", "AUPRC", "Recall@100"]].mean()
              .reset_index())
summary["group"] = pd.Categorical(summary["group"], order, ordered=True)
summary = summary.sort_values(["inference", "relation", "group"])
summary.to_csv(os.path.join(OUT_DIR, "check3x_paper_metrics_summary.csv"), index=False)
for metric in ["AUROC", "AUPRC", "Recall@100"]:
    print(metric)
    print(summary.pivot_table(index=["inference", "relation"], columns="group", values=metric, observed=True)[order].round(3).to_string())
    print()

In [ ]:
# check: the held-out test group should match the benchmark values (data/result/all_checkpoints_evaluation.csv)
bench = pd.read_csv("data/result/all_checkpoints_evaluation.csv")
bench = bench[bench["inference"].isin(RUNS)].groupby(["inference", "edge_type"])[["auc", "auprc", "recall@100"]].mean()
test = summary[summary["group"] == "held-out test edge"].set_index(["inference", "relation"])[["AUROC", "AUPRC", "Recall@100"]]
test.join(bench.rename_axis(["inference", "relation"]).add_prefix("benchmark ")).round(3)